## Setup Cell

- Dataset ("Music_Streaming_Service.db") must be in the same directory.
- Setup cell must run before query cells.

In [2]:
import sqlite3
import pandas as pd
import os
assert os.path.exists("Music_Streaming_Service.db"), f'Missing required file: {"Music_Streaming_Service.db"}'

conn = sqlite3.connect("Music_Streaming_Service.db")
cursor = conn.cursor()
def run_query(query):
    return pd.read_sql_query(query, conn)

### 1. Count of Subscribers per Plan Type

In [3]:
run_query("""
    SELECT sp.type, sp.price, COUNT(us.user_id) AS subscribers
    FROM SubscriptionPlan sp
    LEFT JOIN UserSubscription us ON sp.subscription_id = us.subscription_id
    GROUP BY sp.subscription_id
    ORDER BY subscribers DESC;
""")

,type,price,subscribers
0,Individual Monthly,10.99,35
1,Individual Annual,99.99,18
2,Family Monthly,16.99,6
3,Family Annual,169.99,6


### 2. Top 10 Biggest Playlists and their Authors

In [4]:
run_query("""
    SELECT p.name AS playlist, u.name AS owner, COUNT(ps.song_id) AS songs
    FROM Playlist p
    JOIN User u ON p.user_id = u.user_id
    LEFT JOIN PlaylistSong ps ON p.playlist_id = ps.playlist_id
    GROUP BY p.playlist_id
    ORDER BY songs DESC
    LIMIT 10;
""")

,playlist,owner,songs
0,Coding Focus,Ella Young,25
1,Party Anthems,Liam O'Connor,24
2,Road Trip,Grace Flores,23
3,Workout Mix 2,Ava Davies,23
4,Workout Mix,Carter Turner,22
5,Weekend BBQ 5,Lucas Petro,22
6,Country Classics 1,Elena Rossi,21
7,Chill Vibes,Chloe Dubois,20
8,Study Beats,Ava Davies,20
9,Acoustic Mornings,Grace Flores,20


### 3. Most Eclectic Artists (Artists whose music span the most Genres)

In [5]:
run_query("""
    SELECT ar.name AS artist, COUNT(DISTINCT sg.genre_id) AS genre_count,
       GROUP_CONCAT(DISTINCT g.name) AS genres
    FROM Artist ar
    JOIN Album al ON ar.artist_id = al.artist_id
    JOIN Song s ON al.album_id = s.album_id
    JOIN SongGenre sg ON s.song_id = sg.song_id
    JOIN Genre g ON sg.genre_id = g.genre_id
    GROUP BY ar.artist_id
    ORDER BY genre_count DESC;
""")

,artist,genre_count,genres
0,Rihanna,3,"R&B,Pop,Dance"
1,Michael Jackson,3,"R&B,Pop,Soul"
2,Stevie Nicks,2,"Pop Rock,Rock"
3,Post Malone,2,"Hip Hop,Pop"
4,LeAnn Rimes,2,"Country,Pop"
5,Maroon 5,2,"Pop,Pop Rock"
6,Men at Work,2,"Pop Rock,New Wave"
7,LINKIN PARK,2,"Nu Metal,Alternative Rock"
8,Heart,2,"Hard Rock,Rock"
9,Franz Ferdinand,2,"Alternative Rock,Indie Rock"


### 4. Top 10 Most Eclectic Playlists (Playlists whose music span the most Genres)

In [6]:
run_query("""
    SELECT p.name AS playlist, u.name AS owner,
       COUNT(DISTINCT sg.genre_id) AS unique_genres,
       COUNT(ps.song_id) AS total_songs
    FROM Playlist p
    JOIN User u ON p.user_id = u.user_id
    JOIN PlaylistSong ps ON p.playlist_id = ps.playlist_id
    JOIN SongGenre sg ON ps.song_id = sg.song_id
    GROUP BY p.playlist_id
    ORDER BY unique_genres DESC
    LIMIT 10;
""")

,playlist,owner,unique_genres,total_songs
0,Late Night Drive,Yuki Tanaka,15,28
1,Chill Vibes,Chloe Dubois,14,31
2,Road Trip,Grace Flores,14,33
3,Acoustic Mornings,Grace Flores,14,29
4,Party Anthems,Liam O'Connor,14,36
5,Relaxing Rain,Grace Flores,13,18
6,Heartbreak Hits 2,Maria Garcia,13,27
7,Workout Mix 2,Ava Davies,13,34
8,Relaxing Rain 5,Jacob White,13,23
9,Rock Legends,Elena Rossi,12,19


### 5. Subscription Plans according to Device Type

In [7]:
run_query("""
    SELECT d.type AS device_type, sp.type AS plan, COUNT(DISTINCT u.user_id) AS users
    FROM User u
    JOIN UserDevice ud ON u.user_id = ud.user_id
    JOIN Device d ON ud.device_id = d.device_id
    JOIN UserSubscription us ON u.user_id = us.user_id
    JOIN SubscriptionPlan sp ON us.subscription_id = sp.subscription_id
    GROUP BY d.type, sp.type
    ORDER BY d.type, users DESC;
""")

,device_type,plan,users
0,Desktop,Individual Monthly,10
1,Desktop,Individual Annual,5
2,Desktop,Family Monthly,2
3,Desktop,Family Annual,1
4,Mobile,Individual Monthly,6
5,Mobile,Individual Annual,5
6,Mobile,Family Annual,4
7,Mobile,Family Monthly,2
8,Smart Speaker,Individual Monthly,5
9,Smart Speaker,Individual Annual,4


### 6. Monthly Revenue according to each Subscription Plan

In [8]:
run_query("""
    SELECT sp.type, sp.price, sp.duration_months,
        COUNT(us.user_id) AS subscribers,
        ROUND(COUNT(us.user_id) * sp.price / sp.duration_months, 2) AS monthly_revenue
    FROM SubscriptionPlan sp
    LEFT JOIN UserSubscription us ON sp.subscription_id = us.subscription_id
    GROUP BY sp.subscription_id
    ORDER BY monthly_revenue DESC;
""")

,type,price,duration_months,subscribers,monthly_revenue
0,Individual Monthly,10.99,1,35,384.65
1,Individual Annual,99.99,12,18,149.99
2,Family Monthly,16.99,1,6,101.94
3,Family Annual,169.99,12,6,85.00


### 7. Lifetime Subscription Payment Value per User

In [9]:
run_query("""
    SELECT u.name, u.email,
        COUNT(us.user_subscription_id) AS total_plans,
        ROUND(SUM(sp.price), 2) AS lifetime_value
    FROM User u
    JOIN UserSubscription us ON u.user_id = us.user_id
    JOIN SubscriptionPlan sp ON us.subscription_id = sp.subscription_id
    GROUP BY u.user_id
    ORDER BY lifetime_value DESC;
""")

,name,email,total_plans,lifetime_value
0,Wyatt Campbell,w.camp@example.com,1,169.99
1,Layla Nelson,l.nelson@testmail.org,1,169.99
2,Henry Allen,h.allen@example.com,1,169.99
3,Mason Wright,mason.w@example.com,1,169.99
4,Sofia Müller,sofia.m@testmail.org,1,169.99
5,Maria Garcia,m.garcia@testmail.org,1,169.99
6,Sarah Smith,sarah.s@testmail.org,2,116.98
7,Logan Thomas,l.thomas@example.com,2,110.98
8,Mia Wang,m.wang@testmail.org,2,110.98
9,Isabella Silva,i.silva@testmail.org,2,110.98


### 8. Songs with greater than Average Duration

In [10]:
run_query("""
    SELECT s.title, ar.name AS artist, al.title AS album, s.duration
    FROM Song s
    JOIN Album al ON s.album_id = al.album_id
    JOIN Artist ar ON al.artist_id = ar.artist_id
    WHERE s.duration > (SELECT AVG(duration) FROM Song)
    ORDER BY s.duration DESC;
""")

,title,artist,album,duration
0,Will You Be There,Michael Jackson,Dangerous,7:39
1,The Rolling People,The Verve,Urban Hymns,7:01
2,Dangerous,Michael Jackson,Dangerous,6:59
3,Down by the Sea,Men at Work,Business as Usual,6:53
4,Pink and Velvet,Berlin,Count Three & Pray,6:38
...,...,...,...,...
336,Cheese,Stromae,Cheese,3:02
337,Sunday Girl,Blondie,Parallel Lines,3:01
338,Runaway,Maroon 5,Hands All Over (Deluxe Edition),3:01
339,Sunshine,The All-American Rejects,When the World Comes Down,3:00


### 9. Mobile Only Users + Count of Devices

In [11]:
run_query("""
    SELECT u.name, u.email,
        COUNT(ud.device_id) AS mobile_devices
    FROM User u
    JOIN UserDevice ud ON u.user_id = ud.user_id
    JOIN Device d ON ud.device_id = d.device_id
    GROUP BY u.user_id
    HAVING COUNT(DISTINCT CASE WHEN d.type != 'Mobile' THEN d.device_id END) = 0
    ORDER BY mobile_devices DESC;
""")

,name,email,mobile_devices
0,Layla Nelson,l.nelson@testmail.org,2
1,Julian Stewart,j.stewart@example.com,1
2,Avery Wright,a.wright@testmail.org,1
3,Henry Allen,h.allen@example.com,1
4,Yuki Tanaka,y.tanaka@example.com,1
5,Ahmed Khan,a.khan@example.com,1


### 10. Average Songs per Playlist by Subscription Plan

In [12]:
run_query("""
    SELECT sp.type AS plan,
        COUNT(DISTINCT p.playlist_id) AS total_playlists,
        COUNT(ps.song_id) AS total_songs,
        ROUND(COUNT(ps.song_id) * 1.0 / COUNT(DISTINCT p.playlist_id), 1) AS avg_songs_per_playlist
    FROM SubscriptionPlan sp
    JOIN UserSubscription us ON sp.subscription_id = us.subscription_id
    JOIN Playlist p ON us.user_id = p.user_id
    LEFT JOIN PlaylistSong ps ON p.playlist_id = ps.playlist_id
    GROUP BY sp.type
    ORDER BY avg_songs_per_playlist DESC;
""")

,plan,total_playlists,total_songs,avg_songs_per_playlist
0,Individual Monthly,20,473,23.7
1,Individual Annual,8,115,14.4
2,Family Monthly,2,27,13.5
3,Family Annual,3,36,12.0
